# Diffusion Paper Recreation 1D Version
Recreating the 2D Diffusionmodedl from [HuggingFace](https://huggingface.co/blog/annotated-diffusion) as a 1D Version for electricity timeseries.


In [1]:
%load_ext autoreload
%autoreload 2
%matplotlib widget
from src.Diffusion1d import *
from src.trainloop import *
import numpy as np
import pandas as pd
import joblib
from src.DataConfig import DataConfig
from src.ConditioningDropout import CondDropout
import tempfile, torch
from src.DataPipeline import DataPipeline, make_splits, add_share_cols, hash_checker
from ema_pytorch import EMA
import wandb
wandb.login()
device = "cuda" if torch.cuda.is_available() else "cpu"

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\Samuel\_netrc.
wandb: Currently logged in as: natusvictus (natusvictus-fh-aachen-university-of-applied-sciences) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [2]:
manifest = pd.read_csv(r"../data/manifests/realized_manifest_cleaned.csv", index_col=0)
manifest, appliances = add_appliance_shares(manifest)

In [3]:
strat_cols = ["buildingType", "buildingAgeBin"]
SPLIT_PARAMS1 = dict(holdout_type="TH", holdout_age="DE.05", holdout_year=2021, fractions=[0.7, 0.15, 0.15], seed=42)
SPLIT_PARAMS2 = dict(holdout_type="TH", holdout_age="DE.05", holdout_year=2021, fractions=[0.7, 0.15, 0.15], seed=1874)
train_df = make_splits(manifest, strat_cols, SPLIT_PARAMS1)["train"]
STATIC_CONT_COLS = ["n_persons_sum", "a_ref", "n_apartments"] + add_share_cols(train_df)

buildings -> train 2730, val 585, test 585
rows-> train 8189, val 1755, test 1755
rows-> holdout_building 300, holdout_year 3897, holdout_both 100


### Dataset and Dataloader

In [4]:
# Data Locations
target_root = Path(r"../data/split")

split_dirs = {
    "train": target_root/"train",
    "val": target_root/"val",
    "test": target_root/"test",
    "holdout_year": target_root/"test"/"holdout_year",
    "holdout_building": target_root/"test"/"holdout_building",
    "holdout_both": target_root/"test"/"holdout_both",
}

# Aggregation Method in case of other than 1min Res
TS_AGG = {
    "T": "first",
    "Occupancy Home Active": "first",
    "Occupancy Home Not Active": "first",
    "Electricity Load": "mean",
    "Hot Water Load": "mean",
    "Heating Load": "mean",
    "Off Day": "max"
}

In [5]:
# todo check what happens when no conditioning is used
# Cols to Train on and Generate
TARGET_COLS = ["Electricity Load", "Hot Water Load","Heating Load"] #"T", "Occupancy Home Not Active", "Occupancy Home Not Active","Heating Load", "Electricity Load", "Hot Water Load"

# Timeseries Cols to Condition on
DYNAMIC_COND_COLS = ["Occupancy Home Active", "Occupancy Home Not Active","T","Off Day"] #"T", "Occupancy Home Active", "Occupancy Home Not Active", "T"
CONTEXT_STEPS = 60
CONTEXT_REVERSE = True
N_CTX = len(TARGET_COLS) + 1 if CONTEXT_STEPS > 0 else 0
USE_SEASON = True
N_SEASON = 2 if USE_SEASON else 0
N_DY_CON_COLS = len(DYNAMIC_COND_COLS) + N_SEASON + N_CTX

# Static Parameters to condition on
STATIC_CAT_COLS = ["buildingType","buildingAgeBin","year_type","climateRegion"]
#"buildingType","buildingAgeBin","year_type","climateRegion"
STATIC_BOOL_COLS = [] #"future"


share_cols = [c for c in train_df.columns if c.startswith("apl_")]
share_cols = [c for c in share_cols if train_df[c].std() > 0]
share_cols = list(train_df[share_cols].T.drop_duplicates().index)
STATIC_CONT_COLS_old = ["n_persons_sum", "a_ref","n_apartments"] + share_cols #"a_ref","n_apartments","n_persons_sum


# Cols for normalization of the Target cols so that the load distribution isnt weird across buildings
# define additional normalization by area/occpancy per domain
NORM_METHOD="elec_sh_scale"

norm_variant={
    "none": {},
    "elec_sh_scale": {"Electricity Load": "n_persons_sum","Heating Load": "a_ref"},
    "elec_dhw_sh_scale": {"Electricity Load": "n_persons_sum","Hot Water Load": "n_persons_sum","Heating Load": "a_ref"}
}
NORM_BY_COL = norm_variant[NORM_METHOD]
# all dynamic columns for stat caclculation
columns = TARGET_COLS + [c for c in DYNAMIC_COND_COLS if c not in TARGET_COLS]

ZERO_COLS = []

# define indivdual scalers per column default is zscore
SCALER_BY_COL = {
    "Electricity Load": "log_zscore",
    "Hot Water Load": "log_zscore",
    "Heating Load": "log_zscore",
    "Off Day": "minmax"
}

cond_groups = {
    "occupancy": ["Occupancy Home Active","Occupancy Home Not Active"],
    "temperature": ["T"],
    "off_day": ["Off Day"],
    "season": "season"
}

cond_dropout = {
    "occupancy": 0.2,
    "temperature": 0.3,
    "off_day": 0.1,
    "season": 0.1
}

cond_dropout_all=0.1

static_dropout={"cat": 0.1, "cont": 0.1}

In [6]:
# Parameters for Windowing and Dataloader
WINDOW_DAYS = 1
STRIDE_DAYS = 1
RESOLUTION = "1min"

STEPS_PER_DAY = stepsperday(RESOLUTION)
WINDOW_LENGTH = WINDOW_DAYS * STEPS_PER_DAY
STRIDE = STRIDE_DAYS * STEPS_PER_DAY

In [7]:
cfg = DataConfig(
    target_cols=TARGET_COLS,
    dynamic_cond_cols=DYNAMIC_COND_COLS,
    static_cat_cols=STATIC_CAT_COLS,
    static_cont_cols=STATIC_CONT_COLS,
    static_bool_cols=STATIC_BOOL_COLS,
    resolution=RESOLUTION,
    window_days=WINDOW_DAYS,
    stride_days=STRIDE_DAYS,
    ts_agg=TS_AGG,
    norm_method=NORM_METHOD,
    norm_variant=norm_variant,
    scaler_by_col=SCALER_BY_COL,
    context_steps=CONTEXT_STEPS,
    context_reverse=CONTEXT_REVERSE,
    use_season=USE_SEASON,
    strat_cols=strat_cols,
    split_params=SPLIT_PARAMS1,
    split_dirs={k: str(v) for k, v in split_dirs.items()},
    cond_groups=cond_groups,
    cond_dropout=cond_dropout,
    cond_dropout_all=cond_dropout_all,
    static_dropout=static_dropout
)

In [8]:
cond = CondDropout.from_config(cfg)

In [9]:
cfg.dyn_group_idx

{'occupancy': [0, 1], 'temperature': [2], 'off_day': [3], 'season': [4, 5]}

In [10]:
print(cfg.dyn_group_idx, cfg.dyn_group_probs, cfg.context_mask_idx, cfg.n_dyn_flags, cfg.n_static_flags)

assert DataConfig.from_dict(cfg.to_dict()) == cfg                       # round trip, group order kept
assert list(cfg.dyn_group_idx) == list(DataConfig.from_dict(cfg.to_dict()).dyn_group_idx)

pipe = DataPipeline.build(cfg, manifest)
b = next(iter(pipe.dataloader(pipe.windowdataset("val", pipe.Window_drawer("val", 4, 12)), 8, num_workers=0)))
cond = b["condition"]
assert cond.shape[1] == cfg.n_dyn_cond_cols
K = cfg.context_steps
has_ctx = cond[:, cfg.context_mask_idx, 0] > 0
assert (cond[has_ctx][:, cfg.context_mask_idx, :K] == 1).all()          # the mask channel is where we think it is
assert (cond[:, cfg.context_mask_idx, K:] == 0).all()
print(pipe.model_kwargs())                 # n_dyn_cond_cols should be cond.shape[1] + n_dyn_flags

{'occupancy': [0, 1], 'temperature': [2], 'off_day': [3], 'season': [4, 5]} [0.2, 0.3, 0.1, 0.1] 9 4 1
buildings -> train 2730, val 585, test 585
rows-> train 8189, val 1755, test 1755
rows-> holdout_building 300, holdout_year 3897, holdout_both 100
Missing Files: 7185
Existing Files: 1004
Total: 8189
Missing Files: 1528
Existing Files: 227
Total: 1755
Missing Files: 1531
Existing Files: 224
Total: 1755
Missing Files: 261
Existing Files: 39
Total: 300
Missing Files: 3407
Existing Files: 490
Total: 3897
Missing Files: 84
Existing Files: 16
Total: 100
{'channels': 3, 'cat_cardinalities': [4, 12, 3, 15], 'n_cont': 30, 'n_bool': 1, 'n_dyn_cond_cols': 14}


In [11]:
pipe = DataPipeline.build(cfg, manifest)

buildings -> train 2730, val 585, test 585
rows-> train 8189, val 1755, test 1755
rows-> holdout_building 300, holdout_year 3897, holdout_both 100
Missing Files: 7185
Existing Files: 1004
Total: 8189
Missing Files: 1528
Existing Files: 227
Total: 1755
Missing Files: 1531
Existing Files: 224
Total: 1755
Missing Files: 261
Existing Files: 39
Total: 300
Missing Files: 3407
Existing Files: 490
Total: 3897
Missing Files: 84
Existing Files: 16
Total: 100


In [12]:
preprocessor = pipe.preprocessor
cat_cardinalities = pipe.cat_cardinalities
stats = pipe.stats
target_scaler = pipe.target_scaler
condition_scaler= pipe.condition_scaler
X0_MIN1, X0_MAX1 = pipe.x0_minmax()
train_df_exist = pipe.split_dfs["train"]
val_df_exist = pipe.split_dfs["val"]
test_df_exist = pipe.split_dfs["test"]

train_dataset = pipe.dataset("train", train=True)
val_dataset = pipe.dataset("val")
test_dataset = pipe.dataset("test")

train_df_exist, val_df_exist, test_df_exist = (pipe.split_dfs[s] for s in ("train", "val", "test"))
train_files, train_run_ids, train_static = (pipe.prepare_data("train")[k] for k in ("files", "run_ids", "static"))
val_files, val_run_ids, val_static = (pipe.prepare_data("val")[k] for k in ("files", "run_ids", "static"))
test_files, test_run_ids, test_static = (pipe.prepare_data("test")[k] for k in ("files", "run_ids", "static"))


train_dataset_eval = pipe.windowdataset("train", pipe.Window_drawer("train", 1, 1, seed=0))
val_dataset_eval = pipe.windowdataset("val", pipe.Window_drawer("val", 1, 1, seed=0))
sample_dataset = pipe.windowdataset("val", pipe.Window_drawer("val", 1, 1, seed=0))

In [13]:
BATCH_SIZE = 32
train_dataloader = pipe.dataloader(train_dataset, BATCH_SIZE, train=True)
val_dataloader = pipe.dataloader(val_dataset, BATCH_SIZE)
train_eval_dataloader = pipe.dataloader(train_dataset_eval, BATCH_SIZE)
val_eval_dataloader = pipe.dataloader(val_dataset_eval, BATCH_SIZE)

In [14]:
ds = val_eval_dataloader.dataset
print("windows in index:", len(ds))
print("runs/files:", len(ds.files))
print("batch_size:", val_eval_dataloader.batch_size,
      "| drop_last:", val_eval_dataloader.drop_last,
      "| num_workers:", val_eval_dataloader.num_workers)
print("batches yielded:", sum(1 for _ in val_eval_dataloader))

windows in index: 456
runs/files: 38
batch_size: 32 | drop_last: False | num_workers: 4
batches yielded: 16


In [15]:
schedule = DiffusionSchedule.make("sigmoid", timesteps=1000).to(device)


## Training Loop

In [16]:
EPOCHS = 5
LOSS_TYPE =  "l2"

In [17]:
cond = CondDropout.from_config(cfg)

model_cfg = dict(dim=64,dim_mults=(1, 2, 4),resnet_block_groups=8,**pipe.model_kwargs(),)

optim_cfg = dict(lr=1e-4)
ema_cfg = dict(beta=0.999, update_after_step=100, update_every=3, power=2/3)

model = build_model(model_cfg, device)
optimizer = build_optimizer(model, optim_cfg)
ema = build_ema(model, ema_cfg)
BOUNDARY_WEIGHT = [4.0, 1.0, 10.0]
config = {
    "epochs": EPOCHS, "timesteps": schedule.timesteps, "schedule": "sigmoid",
    "loss_type": LOSS_TYPE, "lr": optimizer.param_groups[0]["lr"],
    "model": model_cfg, "optimizer": optim_cfg, "ema": ema_cfg,
    "data": cfg.to_dict(), "split_hash": pipe.split_hash, "boundary_weight": BOUNDARY_WEIGHT
}
X0_MIN1, X0_MAX1 = pipe.x0_minmax()
extras = {"x0_min": X0_MIN1, "x0_max": X0_MAX1}
config["boundary_weight"] = BOUNDARY_WEIGHT
run_dir = make_run_dir(config, root=r"../outputs/runs")
pipe.save_files(run_dir / "data")
run = wandb.init(project="diffusion-model_3", name=run_dir.name, config=config)
train(model,
      schedule,
      train_dataloader,
      optimizer,
      ema,
      device,
      epochs=EPOCHS,
      loss_type=LOSS_TYPE,
      run=run,
      run_dir=run_dir,
      config=config,
      extras=extras,
      val_dataloader=val_eval_dataloader,
      train_eval_dataloader=train_eval_dataloader,
      val_every_steps=5000,
      val_batches=None,
      ckpt_every_steps=5000,
      val_raw=False,
      boundary_weight=BOUNDARY_WEIGHT,
      cond=cond)
run.finish()

saved data pipeline (split 1015ddb8d77a) to ..\outputs\runs\20261029_210832_54391b\data


epoch 1/5: 0it [00:16, ?it/s]

Loss: 1.4525
Loss: 0.1984
Loss: 0.1736
Loss: 0.1330
Loss: 0.0848
Loss: 0.0912
Loss: 0.0628
Loss: 0.0585
Loss: 0.0998
Loss: 0.0845
Loss: 0.0680
Loss: 0.0839
Loss: 0.0557
Loss: 0.0494
Loss: 0.0575
Loss: 0.0541
Loss: 0.0871
Loss: 0.1042
Loss: 0.0758
Loss: 0.0680
Loss: 0.0503
Loss: 0.0366
Loss: 0.0535
Loss: 0.0596
Loss: 0.0531
Loss: 0.0508
Loss: 0.0779
Loss: 0.0555
Loss: 0.0484
Loss: 0.0489
Loss: 0.0389
Loss: 0.0343
Loss: 0.0537
Loss: 0.0290
Loss: 0.0414
Loss: 0.0446
Loss: 0.0401
Loss: 0.0332
Loss: 0.0701
Loss: 0.0501
Loss: 0.0454
Loss: 0.0495
Loss: 0.0443
Loss: 0.0427
Loss: 0.0359
Loss: 0.0503
Loss: 0.0873
Loss: 0.0523
Loss: 0.0649
Loss: 0.0427
step 5000: val_loss_ema=0.0682
Loss: 0.0516
Loss: 0.0443
Loss: 0.0365
Loss: 0.0397
Loss: 0.0592
Loss: 0.0529
Loss: 0.0520
Loss: 0.0476
Loss: 0.0385
Loss: 0.0520
Loss: 0.0344
Loss: 0.0534
Loss: 0.0474
Loss: 0.0370
Loss: 0.0413
Loss: 0.0390
Loss: 0.0446
Loss: 0.0271
Loss: 0.0378
Loss: 0.0283
Loss: 0.0408
Loss: 0.0420
Loss: 0.0458
Loss: 0.0334
Loss: 0

epoch 2/5: 0it [00:00, ?it/s]

Loss: 0.0350
Loss: 0.0273
Loss: 0.0311
Loss: 0.0453
Loss: 0.0267
Loss: 0.0514
Loss: 0.0403
Loss: 0.0364
Loss: 0.0440
Loss: 0.0473
Loss: 0.0305
Loss: 0.0434
Loss: 0.0465
Loss: 0.0528
Loss: 0.0543
Loss: 0.0422
Loss: 0.0363
Loss: 0.0447
Loss: 0.0514
Loss: 0.0216
Loss: 0.0293
Loss: 0.0427
Loss: 0.0519
Loss: 0.0369
Loss: 0.0411
Loss: 0.0327
Loss: 0.0390
Loss: 0.0644
Loss: 0.0438
Loss: 0.0439
Loss: 0.0468
Loss: 0.0298
Loss: 0.0532
Loss: 0.0407
Loss: 0.0330
Loss: 0.0333
step 15000: val_loss_ema=0.0509
Loss: 0.0437
Loss: 0.0414
Loss: 0.0480
Loss: 0.0503
Loss: 0.0374
Loss: 0.0529
Loss: 0.0403
Loss: 0.0471
Loss: 0.0555
Loss: 0.0338
Loss: 0.0367
Loss: 0.0445
Loss: 0.0594
Loss: 0.0440
Loss: 0.0403
Loss: 0.0447
Loss: 0.0372
Loss: 0.0436
Loss: 0.0288
Loss: 0.0454
Loss: 0.0541
Loss: 0.0493
Loss: 0.0467
Loss: 0.0483
Loss: 0.0411
Loss: 0.0377
Loss: 0.0438
Loss: 0.0310
Loss: 0.0370
Loss: 0.0511
Loss: 0.0462
Loss: 0.0443
Loss: 0.0333
Loss: 0.0431
Loss: 0.0438
Loss: 0.0381
Loss: 0.0395
Loss: 0.0381
Loss: 

...

epoch 4/5: 0it [00:00, ?it/s]

Loss: 0.0506
Loss: 0.0495
Loss: 0.0363
Loss: 0.0438
Loss: 0.0444
Loss: 0.0213
Loss: 0.0475
step 35000: val_loss_ema=0.0455
Loss: 0.0376
Loss: 0.0347
Loss: 0.0334
Loss: 0.0321
Loss: 0.0415
Loss: 0.0479
Loss: 0.0370
Loss: 0.0448
Loss: 0.0279
Loss: 0.0380
Loss: 0.0373
Loss: 0.0361
Loss: 0.0328
Loss: 0.0441
Loss: 0.0444
Loss: 0.0476
Loss: 0.0372
Loss: 0.0494
Loss: 0.0304
Loss: 0.0294
Loss: 0.0350
Loss: 0.0381
Loss: 0.0382
Loss: 0.0437
Loss: 0.0373
Loss: 0.0380
Loss: 0.0372
Loss: 0.0475
Loss: 0.0368
Loss: 0.0311
Loss: 0.0336
Loss: 0.0482
Loss: 0.0432
Loss: 0.0437
Loss: 0.0331
Loss: 0.0373
Loss: 0.0388
Loss: 0.0381
Loss: 0.0438
Loss: 0.0536
Loss: 0.0389
Loss: 0.0436
Loss: 0.0383
Loss: 0.0296
Loss: 0.0401
Loss: 0.0289
Loss: 0.0343
Loss: 0.0369
Loss: 0.0304
Loss: 0.0389
step 40000: val_loss_ema=0.0449
Loss: 0.0360
Loss: 0.0409
Loss: 0.0341
Loss: 0.0300
Loss: 0.0322
Loss: 0.0323
Loss: 0.0387
Loss: 0.0226
Loss: 0.0338
Loss: 0.0325
Loss: 0.0367
Loss: 0.0407
Loss: 0.0316
Loss: 0.0474
Loss: 0.0345


epoch 5/5: 0it [00:00, ?it/s]

Loss: 0.0386
Loss: 0.0436
Loss: 0.0353
Loss: 0.0378
Loss: 0.0328
Loss: 0.0347
Loss: 0.0374
Loss: 0.0512
Loss: 0.0307
Loss: 0.0325
Loss: 0.0318
Loss: 0.0345
Loss: 0.0294
Loss: 0.0394
Loss: 0.0444
Loss: 0.0371
Loss: 0.0321
Loss: 0.0260
Loss: 0.0399
Loss: 0.0386
Loss: 0.0644
Loss: 0.0610
Loss: 0.0290
Loss: 0.0332
Loss: 0.0387
Loss: 0.0530
Loss: 0.0333
Loss: 0.0515
Loss: 0.0361
Loss: 0.0400
Loss: 0.0504
Loss: 0.0387
Loss: 0.0559
Loss: 0.0428
Loss: 0.0405
Loss: 0.0375
Loss: 0.0449
Loss: 0.0363
Loss: 0.0312
Loss: 0.0294
Loss: 0.0424
Loss: 0.0426
Loss: 0.0395
step 50000: val_loss_ema=0.0440
Loss: 0.0405
Loss: 0.0425
Loss: 0.0338
Loss: 0.0609
Loss: 0.0432
Loss: 0.0630
Loss: 0.0287
Loss: 0.0339
Loss: 0.0499
Loss: 0.0490
Loss: 0.0464
Loss: 0.0400
Loss: 0.0413
Loss: 0.0266
Loss: 0.0303
Loss: 0.0382
Loss: 0.0344
Loss: 0.0320
Loss: 0.0373
Loss: 0.0420
Loss: 0.0383
Loss: 0.0282
Loss: 0.0263
Loss: 0.0432
Loss: 0.0552
Loss: 0.0413
Loss: 0.0375
Loss: 0.0349
Loss: 0.0212
Loss: 0.0299
Loss: 0.0298
Loss: 

epoch,▁▁▁▁▁▁▁▁▃▃▃▃▃▃▃▃▃▃▃▅▅▅▅▆▆▆▆▆▆▆▆▆▆▆██████
epoch/loss,█▂▂▁▁
global_step,▁▁▁▁▁▂▂▂▂▂▂▂▃▃▃▃▃▃▃▄▄▄▄▄▄▅▅▆▆▆▆▆▆▆▆▇▇▇▇█
train/learning_rate,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/loss,▅▁▂█▄▅▃▃▆▅▄▃▃▃▃▄▃▃▆▂▄▂▄▁▂▃▅▃▃▃▃▃▂▅▃▂▄▄▂▂
train_eval/loss_ema,█▄▃▂▂▂▂▁▁▁▁
train_eval/loss_ema_t0,█▄▃▂▂▂▁▁▁▁▁
train_eval/loss_ema_t111,█▅▄▃▃▂▂▂▁▁▁
train_eval/loss_ema_t222,█▅▄▃▃▂▂▂▁▁▁
train_eval/loss_ema_t333,█▅▄▃▃▂▂▂▁▁▁
+17,...
